In [3]:
"""
Script to generate continuous sequences for Phase 2 from Phase 1 isolated signs.
Uses REAL keypoints from Phase 1 and concatenates them with smooth transitions.
Cuts preparation and ending frames for more natural continuous sequences.
"""
import os
import json
import random
import numpy as np
from typing import List, Tuple, Dict

# Configuration
PHASE1_TRAIN_DIR = 'Data_Keypoints_holistic/train'
PHASE1_VAL_DIR = 'Data_Keypoints_holistic/val'

OUTPUT_TRAIN_DIR = 'Data_Keypoints_holistic/continuous_train'
OUTPUT_VAL_DIR = 'Data_Keypoints_holistic/continuous_val'

NUM_TRAIN_SAMPLES = 2400
NUM_VAL_SAMPLES = 300

NUM_KEYPOINTS = 141
MIN_SIGNS_PER_SEQUENCE = 2
MAX_SIGNS_PER_SEQUENCE = 6
TRANSITION_FRAMES = 10  # Smooth transition between signs
CUT_START_FRAMES = 10   # Cut preparation phase
CUT_END_FRAMES = 5      # Cut ending phase

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


def load_phase1_data(data_dir: str) -> Tuple[Dict[str, List[np.ndarray]], List[str]]:
    """Load all isolated sign sequences from Phase 1."""
    if not os.path.exists(data_dir):
        raise FileNotFoundError(f'Phase 1 data not found: {data_dir}')
    
    class_names = sorted([
        name for name in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, name))
    ])
    
    if not class_names:
        raise ValueError(f'No classes found in {data_dir}')
    
    print(f"Loading from: {data_dir}")
    print(f"Found {len(class_names)} classes")
    
    data_dict = {}
    total = 0
    
    for class_name in class_names:
        class_dir = os.path.join(data_dir, class_name)
        sequences = []
        
        for file_name in os.listdir(class_dir):
            if not file_name.endswith('.npy'):
                continue
            
            try:
                seq = np.load(os.path.join(class_dir, file_name)).astype(np.float32)
                if seq.ndim != 2:
                    seq = seq.reshape(seq.shape[0], -1)
                if seq.shape[1] == NUM_KEYPOINTS:
                    sequences.append(seq)
                    total += 1
            except:
                pass
        
        if sequences:
            data_dict[class_name] = sequences
            print(f"  {class_name}: {len(sequences)} sequences")
    
    print(f"Total: {total} sequences\n")
    return data_dict, class_names


def cut_sequence(seq: np.ndarray, cut_start: int, cut_end: int) -> np.ndarray:
    """Cut preparation and ending frames."""
    n = len(seq)
    if cut_start + cut_end >= n - 10:
        start, end = int(n * 0.2), int(n * 0.8)
        return seq[start:end]
    return seq[cut_start:n-cut_end]


def create_transition(seq1: np.ndarray, seq2: np.ndarray, n_frames: int) -> np.ndarray:
    """Create smooth transition between sequences."""
    if len(seq1) == 0 or len(seq2) == 0:
        return np.zeros((n_frames, NUM_KEYPOINTS), dtype=np.float32)
    
    start, end = seq1[-1], seq2[0]
    trans = np.zeros((n_frames, NUM_KEYPOINTS), dtype=np.float32)
    
    for i in range(n_frames):
        alpha = (i + 1) / (n_frames + 1)
        trans[i] = (1 - alpha) * start + alpha * end
    
    return trans


def concatenate_signs(data_dict: Dict, class_names: List[str], labels: List[int]) -> np.ndarray:
    """Concatenate multiple signs into continuous sequence."""
    sequences = []
    
    for label in labels:
        class_name = class_names[label]
        if class_name not in data_dict or not data_dict[class_name]:
            continue
        
        seq = random.choice(data_dict[class_name]).copy()
        cut_seq = cut_sequence(seq, CUT_START_FRAMES, CUT_END_FRAMES)
        
        if len(cut_seq) >= 5:
            sequences.append(cut_seq)
    
    if not sequences:
        raise ValueError("No valid sequences")
    
    result = sequences[0]
    for i in range(1, len(sequences)):
        trans = create_transition(result, sequences[i], TRANSITION_FRAMES)
        result = np.concatenate([result, trans, sequences[i]], axis=0)
    
    return result


def generate_sample(sample_id: int, output_dir: str, data_dict: Dict, class_names: List[str]) -> bool:
    """Generate one continuous sample."""
    try:
        num_signs = random.randint(MIN_SIGNS_PER_SEQUENCE, MAX_SIGNS_PER_SEQUENCE)
        
        labels = []
        prev = -1
        for _ in range(num_signs):
            available = [i for i in range(len(class_names)) if i != prev]
            label = random.choice(available if available else list(range(len(class_names))))
            labels.append(label)
            prev = label
        
        keypoints = concatenate_signs(data_dict, class_names, labels)
        text = ' '.join([class_names[i] for i in labels])
        
        np.save(os.path.join(output_dir, f'sample_{sample_id:03d}.npy'), keypoints)
        
        with open(os.path.join(output_dir, f'sample_{sample_id:03d}_labels.json'), 'w') as f:
            json.dump({
                'labels': labels,
                'text': text,
                'num_frames': len(keypoints),
                'num_signs': len(labels)
            }, f, indent=2)
        
        return True
    except:
        return False


def generate_dataset(output_dir: str, n_samples: int, data_dict: Dict, class_names: List[str]):
    """Generate complete dataset."""
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Generating {n_samples} samples...")
    success = sum(1 for i in range(n_samples) if generate_sample(i+1, output_dir, data_dict, class_names))
    
    with open(os.path.join(output_dir, 'class_names.json'), 'w') as f:
        json.dump({'classes': class_names}, f, indent=2)
    
    print(f"Generated {success}/{n_samples} samples\n")


def verify_dataset(dataset_dir: str):
    """Verify dataset."""
    print(f"Verifying {dataset_dir}...")
    
    if not os.path.exists(os.path.join(dataset_dir, 'class_names.json')):
        print("Missing class_names.json")
        return
    
    npy_files = [f for f in os.listdir(dataset_dir) if f.endswith('.npy')]
    json_files = [f for f in os.listdir(dataset_dir) if f.endswith('_labels.json')]
    
    print(f"{len(npy_files)} .npy files")
    print(f"{len(json_files)} label files")
    
    if json_files:
        frames, signs = [], []
        for jf in json_files:
            with open(os.path.join(dataset_dir, jf)) as f:
                data = json.load(f)
                frames.append(data['num_frames'])
                signs.append(data['num_signs'])
        
        print(f"Avg frames: {np.mean(frames):.1f}, Avg signs: {np.mean(signs):.1f}\n")


def main():
    """Main function."""
    print("Generate Continuous Sequences From Phase 1 Isolated Sign")
    print(f"\nConfiguration:")
    print(f"  Train samples: {NUM_TRAIN_SAMPLES}")
    print(f"  Val samples: {NUM_VAL_SAMPLES}")
    print(f"  Signs per sequence: {MIN_SIGNS_PER_SEQUENCE}-{MAX_SIGNS_PER_SEQUENCE}")
    print(f"  Cut start: {CUT_START_FRAMES} frames (remove preparation)")
    print(f"  Cut end: {CUT_END_FRAMES} frames (remove ending)")
    print(f"  Transition: {TRANSITION_FRAMES} frames (smooth interpolation)\n")
    
    try:
        print("Loading Phase 1 Training Data...")
        train_dict, train_classes = load_phase1_data(PHASE1_TRAIN_DIR)
        
        print("Loading Phase 1 Validation Data...")
        val_dict, val_classes = load_phase1_data(PHASE1_VAL_DIR)
        
        print("Generating Training Set...")
        generate_dataset(OUTPUT_TRAIN_DIR, NUM_TRAIN_SAMPLES, train_dict, train_classes)
        
        print("Generating Validation Set...")
        generate_dataset(OUTPUT_VAL_DIR, NUM_VAL_SAMPLES, val_dict, val_classes)
        
        print("Verification")
        verify_dataset(OUTPUT_TRAIN_DIR)
        verify_dataset(OUTPUT_VAL_DIR)
        
        print("Complete!")
        print(f"\nOutput:")
        print(f"  Train: {OUTPUT_TRAIN_DIR}/")
        print(f"  Val: {OUTPUT_VAL_DIR}/")
        print(f"\nFeatures:")
        print(f"  Uses REAL keypoints from Phase 1")
        print(f"  Cuts preparation phase ({CUT_START_FRAMES} frames)")
        print(f"  Cuts ending phase ({CUT_END_FRAMES} frames)")
        print(f"  Smooth transitions ({TRANSITION_FRAMES} frames)")
        print(f"\nReady for training:")
        print(f"  python Train_Model_Phase_2.py")
        
    except Exception as e:
        print(f"\nError: {e}")
        print("\nPlease ensure Phase 1 data exists in:")
        print(f"  {PHASE1_TRAIN_DIR}")
        print(f"  {PHASE1_VAL_DIR}")


if __name__ == "__main__":
    main()


Generate Continuous Sequences From Phase 1 Isolated Sign

Configuration:
  Train samples: 2400
  Val samples: 300
  Signs per sequence: 2-6
  Cut start: 10 frames (remove preparation)
  Cut end: 5 frames (remove ending)
  Transition: 10 frames (smooth interpolation)

Loading Phase 1 Training Data...
Loading from: Data_Keypoints_holistic/train
Found 30 classes
  Accept: 80 sequences
  Buy: 80 sequences
  Call: 80 sequences
  Candy: 80 sequences
  Catch: 80 sequences
  Deaf: 80 sequences
  Everyone: 80 sequences
  Food: 80 sequences
  Give: 80 sequences
  Green: 80 sequences
  Help: 80 sequences
  Hungry: 80 sequences
  I: 80 sequences
  Learn: 80 sequences
  Light-blue: 80 sequences
  Like: 80 sequences
  Milk: 80 sequences
  Music: 80 sequences
  Name: 80 sequences
  Red: 80 sequences
  Ship: 80 sequences
  Son: 80 sequences
  Thanks: 80 sequences
  Want: 80 sequences
  Water: 80 sequences
  Where: 80 sequences
  Women: 80 sequences
  Yellow: 80 sequences
  Yogurt: 80 sequences
  You: 